# Insertion, selections, and rigid transforms

`Recipe.insert()` records a manufacturing activation operation. The
collection is transformed once, packed into the eventual device world,
and remains dormant until its operation is reached.

In [ ]:
import tangle

# This collection is authored around a local origin and is not yet in
# the periodic simulation cell.
cell = tangle.Cell([1e-3, 1e-3, 2e-3], periodic=[True, True, False])
material = tangle.Material("fiber", diameter=19e-6)
collection = tangle.FiberCollection.from_centerlines(
    [[[-0.3e-3, 0.0, 0.0], [0.3e-3, 0.0, 0.0]]],
    material,
    name="local-coordinate ply",
    formation_layer=0,
)
# 0=x, 1=y, 2=z; z is the layer stacking direction here.
recipe = tangle.Recipe(cell, layer_axis=2)
# Rotation is applied first, then translation places the rotated fiber.
selection = recipe.insert(
    collection,
    name="placed ply",
    translation=[0.5e-3, 0.5e-3, 0.4e-3],
    rotation=[[0.0, -1.0, 0.0],
              [1.0,  0.0, 0.0],
              [0.0,  0.0, 1.0]],
)
print(selection.name, selection.fiber_ids, selection.formation_step)

`name` labels the returned `FiberSelection`; `translation` is in meters;
and `rotation` is a 3×3 matrix applied before translation. Selections are
stable handles for reporting and future selection-scoped APIs. Use
`recipe.operations()` to audit ordering before a costly run and
`recipe.centerlines()` to inspect the packed initial geometry.

In [ ]:
# Methods append ordered operations; no solver launches until run().
recipe.relax(maximum_iterations=2_000)
print(*recipe.operations(), sep="\n")
recipe.centerlines()